## Logistics Simulation & What-If Scenarios

In [ ]:
import numpy as np
import pandas as pd
import yaml
from pathlib import Path

# reload config (you edited config.yaml)
with open("../config.yaml", "r") as f:
    cfg = yaml.safe_load(f)

# load processed data created in Steps 3 & 4
lanes   = pd.read_csv("../data/processed/lanes_generated.csv")
demand  = pd.read_csv("../data/processed/demand_simulated.csv")

# quick sanity checks
print("lanes:", lanes.shape)
print("demand:", demand.shape)
assert {"retailer_id","sku_id","week","demand_units"}.issubset(demand.columns)
assert {"from_type","to_type","base_lead_days","transport_cost_per_unit"}.issubset(lanes.columns)

In [ ]:
def simulate_shipment(retailer_id, sku_id, week, units, lanes_df, reliability_cfg):
    """
    Simulate one shipment flow from supplier -> factory -> warehouse -> retailer
    Returns total_lead_days, delivered_on_time (bool), total_cost
    """
    total_lead = 0
    total_cost = 0
    route = ["supplier", "factory", "warehouse", "retail"]

    # Each leg contributes base lead and cost
    for i in range(len(route) - 1):
        leg_from, leg_to = route[i], route[i + 1]
        lane = lanes_df[(lanes_df.from_type == leg_from) & (lanes_df.to_type == leg_to)].sample(1).iloc[0]
        base_lt = lane["base_lead_days"]
        cost = lane["transport_cost_per_unit"] * units

        # Apply reliability: sometimes delay occurs
        if np.random.rand() > reliability_cfg["lane_reliability"]:
            delay = np.random.randint(1, reliability_cfg["max_delay_days"] + 1)
        else:
            delay = 0

        total_lead += base_lt + delay
        total_cost += cost

    # On-time if total_lead <= expected (mean lead_time * 1.1)
    expected_lead =  (len(route) - 1) * reliability_cfg["expected_base_days"]
    on_time = total_lead <= expected_lead * 1.1
    return total_lead, on_time, total_cost

In [ ]:
rows = []
reliability_cfg = {
    "lane_reliability": cfg["lead_time"]["reliability_default"],
    "expected_base_days": int(lanes["base_lead_days"].mean()),
    "max_delay_days": cfg["lead_time"]["max_delay_days"]
}

for _, r in demand.iterrows():
    lead, ontime, cost = simulate_shipment(
        r["retailer_id"], r["sku_id"], r["week"], r["demand_units"],
        lanes, reliability_cfg
    )
    rows.append({
        "retailer_id": r["retailer_id"],
        "sku_id": r["sku_id"],
        "week": r["week"],
        "demand_units": r["demand_units"],
        "realized_lead_days": lead,
        "delivered_on_time": ontime,
        "transport_cost": cost
    })

ship_df = pd.DataFrame(rows)
ship_df.head()

In [ ]:
summary = {
    "avg_lead_days": round(ship_df["realized_lead_days"].mean(), 2),
    "on_time_rate": round(ship_df["delivered_on_time"].mean() * 100, 2),
    "total_cost": round(ship_df["transport_cost"].sum(), 2)
}
print(summary)

In [ ]:
output_path = Path("../data/processed/shipments_simulated.csv")
ship_df.to_csv(output_path, index=False)
print(f"✅ Shipment simulation saved to {output_path}")